In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from scipy.spatial import cKDTree

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'philadelphia'
STATE_FIPS = '42'
COUNTY_FIPS = '101'
MODEL_TYPE = 'split_rate_4to1_lycd_post_abatement'
LAND_IMPROVEMENT_RATIO = 4.0
MILLAGE = 13.998  # combined city (0.6317%) + school district (0.7681%) = 1.3998%

# GMA zone assignment: static reference extracted from OPA's 2025 GMA PDF
# (parcel centroid → L1/L2/L3 zone labels; 17 / 84 / 613 zones)
GMA_PATH = Path('data/parcel_gma_assignment.parquet')
DOR_SHP_PATH = Path(
    r'C:\projects\LVTShift\cities\philadelphia\data'
    r'\dor_parcels\Philadelphia_DOR_Parcels_202402.shp'
)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)


## Step 1: Load parcel data

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
gdf = gpd.read_parquet(PARCEL_PATH)
gdf['parcel_number'] = gdf['parcel_number'].astype(str).str.zfill(9)
print(f'Loaded {len(gdf):,} parcels')

Loaded 579,814 parcels


## Step 2: Lot area — DOR PIN-keyed lookup (primary) + fallbacks

Each OPA parcel has a `pin` field (10-digit DOR format) matching the `PIN` column in
the Philadelphia DOR Parcels GeoJSON. We pre-built `parcel_areas_by_pin.parquet`
(577K entries) mapping each DOR PIN to its polygon area in sqft — no spatial join.
This eliminates the artifact where multiple OPA centroid points fall inside one large
DOR polygon and inherit its full area.

Fallback chain: PIN-keyed area → OPA `total_area` → DOR spatial join cache → KNN.

In [3]:
PIN_AREA_PATH = DATA_DIR / 'parcel_areas_by_pin.parquet'
AREA_PATH     = DATA_DIR / 'parcel_areas_dor.parquet'

pin_areas = pd.read_parquet(PIN_AREA_PATH)
pin_areas['pin'] = pin_areas['pin'].astype(str).str.strip()
print(f'PIN-area lookup:   {len(pin_areas):,} unique polygons')

if AREA_PATH.exists():
    dor_areas = pd.read_parquet(AREA_PATH)
    print(f'DOR spatial cache: {len(dor_areas):,} rows (fallback only)')
else:
    print('Building DOR spatial join cache (one-time, ~30s)...')
    dor = gpd.read_file(DOR_SHP_PATH, columns=['Shape__Are'])
    pts = gpd.read_parquet(PARCEL_PATH, columns=['parcel_number', 'geometry'])
    pts = pts.to_crs('EPSG:3857')
    joined = gpd.sjoin(pts, dor[['Shape__Are', 'geometry']], how='left', predicate='within')
    joined['dor_area_sqft'] = (joined['Shape__Are'] * 10.7639).round(1)
    dor_areas = joined[['parcel_number', 'dor_area_sqft']].copy()
    dor_areas.to_parquet(AREA_PATH, index=False)
    print(f'  Cached {len(dor_areas):,} rows')


PIN-area lookup:   577,909 unique polygons
DOR spatial cache: 579,815 rows (fallback only)


## Step 3: Load GMA zone assignments and join lot area to parcels

In [4]:
gma = pd.read_parquet(GMA_PATH)
gma['key'] = gma['key'].astype(str).str.zfill(9)
print(f'GMA assignment: {len(gma):,} parcels | '
      f'L1={gma["gma1"].nunique()} zones | L2={gma["gma2"].nunique()} | L3={gma["gma3"].nunique()}')

# --- Lot area: three-source priority ---
# 1. OPA total_area  — direct, one-to-one, no spatial artifact possible
# 2. PIN-keyed DOR polygon area — for the ~32K with total_area = 0 that have a PIN
# 3. DOR spatial join — last resort (may assign shared campus polygon)
gdf['pin'] = gdf['pin'].astype(str).str.strip()
dor_areas['parcel_number'] = dor_areas['parcel_number'].astype(str).str.zfill(9)

_opa_area  = pd.to_numeric(gdf['total_area'], errors='coerce').fillna(0)
_pin_map   = pin_areas.drop_duplicates('pin').set_index('pin')['pin_area_sqft']
_pin_area  = gdf['pin'].map(_pin_map)
_dor_map   = dor_areas.drop_duplicates('parcel_number').set_index('parcel_number')['dor_area_sqft']
_dor_area  = gdf['parcel_number'].map(_dor_map)

_s1 = _opa_area > 0                                            # OPA total_area
_s2 = ~_s1 & _pin_area.notna() & (_pin_area > 0)             # PIN-keyed DOR
_s3 = ~_s1 & ~_s2 & _dor_area.notna() & (_dor_area > 0)     # DOR spatial

gdf['dor_area_sqft'] = np.where(_s1, _opa_area,
                        np.where(_s2, _pin_area, _dor_area))

print(f'Lot area source — OPA total_area: {_s1.sum():,} | PIN-keyed DOR: {_s2.sum():,} | DOR spatial: {_s3.sum():,}')
print(f'Parcels needing area KNN:         {(~_s1 & ~_s2 & ~_s3).sum():,}')

# Join GMA zone labels
gdf = gdf.merge(
    gma[['key', 'gma1', 'gma2', 'gma3']],
    left_on='parcel_number', right_on='key',
    how='left',
)

gma_matched  = gdf['gma3'].notna()
area_matched = gdf['dor_area_sqft'].notna() & (gdf['dor_area_sqft'] > 0)
print(f'\nParcels with GMA assignment:      {gma_matched.sum():,} ({gma_matched.mean():.1%})')
print(f'Parcels with lot area:             {area_matched.sum():,} ({area_matched.mean():.1%})')
print(f'Parcels needing GMA KNN fallback:  {(~gma_matched).sum():,}')
print(f'Parcels needing area imputation:   {(~area_matched).sum():,}')


GMA assignment: 528,204 parcels | L1=17 zones | L2=84 | L3=613


Lot area source — OPA total_area: 547,945 | PIN-keyed DOR: 1,220 | DOR spatial: 29,742
Parcels needing area KNN:         907



Parcels with GMA assignment:      525,544 (90.6%)
Parcels with lot area:             578,907 (99.8%)
Parcels needing GMA KNN fallback:  54,270
Parcels needing area imputation:   907


## Step 4: Compute GMA hierarchical LYCD land values

For each parcel, land value is estimated using the "Least You Can Do" (LYCD) method
applied at OPA's Geographic Market Area (GMA) zone level:

1. For each GMA zone at the finest applicable level (L3 if ≥ 50 improved parcels,
   else L2, else L1): compute `median(market_value / dor_area_sqft)` over **improved**
   parcels in that zone. "Improved" = OPA category code not in {6, 12, 13}.
2. Apply land allocation: 20% for improved parcels (OPA's standard), 100% for vacant.
3. `lycd_land_value = zone_land_psf × parcel_dor_area_sqft`.

Using `market_value` from the same `assessments WHERE year=2024` pull as the billing
values ensures vintage consistency. The GMA hierarchy (L1: 17 zones, L2: 84 zones,
L3: 613 micro-zones) is OPA's own internal geography for setting assessments.

KNN fallback (5 nearest neighbors) is applied for parcels without a GMA assignment
or without a DOR lot area.

In [5]:
LAND_PCT_IMPROVED = 0.20   # OPA's standard land allocation for improved parcels
LAND_PCT_VACANT   = 1.00   # vacant parcels: market value is all land
MIN_IMPROVED      = 50     # minimum improved parcels per GMA zone before falling back

# Improved vs vacant flag using OPA category codes (same codes as the four-override system)
VACANT_CODES = {'6', '12', '13'}
cat_raw = gdf['category_code'].astype(str).str.strip()
has_market   = gdf['market_value'].notna() & (gdf['market_value'] > 0)
has_area     = gdf['dor_area_sqft'].notna() & (gdf['dor_area_sqft'] > 0)
is_improved  = ~cat_raw.isin(VACANT_CODES) & has_market & has_area
is_vacant_p  =  cat_raw.isin(VACANT_CODES) & has_market & has_area

# Per-parcel total-value PSF (used only to build zone medians for improved parcels)
gdf['_total_psf'] = np.where(
    is_improved | is_vacant_p,
    gdf['market_value'] / gdf['dor_area_sqft'],
    np.nan,
)

# Zone medians of total PSF over improved parcels at each GMA level
imp = gdf[is_improved]
l3_cnt = imp.groupby('gma3')['_total_psf'].count().rename('_l3_n')
l2_cnt = imp.groupby('gma2')['_total_psf'].count().rename('_l2_n')
l3_med = imp.groupby('gma3')['_total_psf'].median().rename('_l3_med')
l2_med = imp.groupby('gma2')['_total_psf'].median().rename('_l2_med')
l1_med = imp.groupby('gma1')['_total_psf'].median().rename('_l1_med')

for col_key, series in [('gma3', l3_cnt), ('gma3', l3_med),
                         ('gma2', l2_cnt), ('gma2', l2_med),
                         ('gma1', l1_med)]:
    gdf = gdf.merge(series.reset_index(), on=col_key, how='left')

# Hierarchical zone-median selection
gdf['_gma_med'] = np.where(
    gdf['_l3_n'].fillna(0) >= MIN_IMPROVED, gdf['_l3_med'],
    np.where(gdf['_l2_n'].fillna(0) >= MIN_IMPROVED, gdf['_l2_med'], gdf['_l1_med'])
)

# Apply land allocation per parcel type
_land_pct = np.where(is_vacant_p, LAND_PCT_VACANT, LAND_PCT_IMPROVED)
_land_psf  = gdf['_gma_med'] * _land_pct

# Land value for GMA-assigned parcels with a DOR area
gdf['lycd_land_value'] = np.where(
    gdf['gma3'].notna() & has_area,
    (_land_psf * gdf['dor_area_sqft']).clip(lower=0),
    np.nan,
)

# Track GMA level used
gdf['gma_level'] = np.where(
    gdf['gma3'].isna(), 'knn',
    np.where(gdf['_l3_n'].fillna(0) >= MIN_IMPROVED, 'L3',
    np.where(gdf['_l2_n'].fillna(0) >= MIN_IMPROVED, 'L2', 'L1'))
)

n_gma       = int(gdf['lycd_land_value'].notna().sum())
n_knn_needed = int(gdf['lycd_land_value'].isna().sum())
print(f'GMA land values computed:      {n_gma:,}')
print(f'Parcels needing KNN fallback:  {n_knn_needed:,}')
print()

# Project to EPSG:3857 for KNN
gdf_m  = gdf.to_crs('EPSG:3857')
coords = np.column_stack([gdf_m.geometry.x, gdf_m.geometry.y])

def knn_impute(values, coords, K=5):
    values = np.asarray(values, dtype=float)
    has_v  = ~np.isnan(values)
    need_v = np.isnan(values)
    if need_v.sum() == 0:
        return values
    tree = cKDTree(coords[has_v])
    _, idxs = tree.query(coords[need_v], k=min(K, int(has_v.sum())), workers=-1)
    out = values.copy()
    out[need_v] = np.median(values[has_v][idxs], axis=1)
    return out

# KNN-impute DOR area for parcels without a polygon hit
n_area_missing = int(gdf['dor_area_sqft'].isna().sum())
gdf['dor_area_sqft'] = knn_impute(gdf['dor_area_sqft'].values.astype(float), coords)
print(f'KNN-imputed dor_area_sqft for {n_area_missing:,} parcels')

# KNN-impute lycd_land_value for parcels outside GMA coverage
gdf['lycd_land_value'] = knn_impute(gdf['lycd_land_value'].values, coords)
print(f'KNN-imputed lycd_land_value for {n_knn_needed:,} parcels (no GMA zone or DOR area)')

gdf.loc[gdf['gma_level'] == 'knn', 'gma_level'] = 'knn'






# Cap lycd_land_value at market_value for improved parcels (non-vacant OPA codes).
# Vacant parcels are exempt from the cap: OPA systematically undervalues them and
# the LYCD rate (100% zone PSF) reflects their development potential.
# The cap is a principled safety net: land value cannot exceed total property value.
# It handles parcels — like most OPA code 8 (accessory structures) — that fall back
# to the DOR spatial join and inevitably receive inflated shared-polygon areas.
_is_vac_cap = gdf['category_code'].astype(str).str.strip().isin({'6', '12', '13'})
_mv_cap = pd.to_numeric(gdf['market_value'], errors='coerce').fillna(0).clip(lower=0)
_n_capped = int(((gdf['lycd_land_value'] > _mv_cap) & ~_is_vac_cap).sum())
gdf['lycd_land_value'] = np.where(
    ~_is_vac_cap,
    np.minimum(gdf['lycd_land_value'], _mv_cap),
    gdf['lycd_land_value'],
)
print(f'Capped lycd_land_value at market_value (improved parcels only): {_n_capped:,}')
# Clean up temporaries
gdf.drop(columns=['_total_psf', '_gma_med', '_l3_n', '_l2_n',
                   '_l3_med', '_l2_med', '_l1_med'], inplace=True)

print()
print('GMA level used:')
print(gdf['gma_level'].value_counts().to_string())
print()
print(f'All parcels have lycd_land_value: {gdf["lycd_land_value"].isna().sum() == 0}')


GMA land values computed:      525,544
Parcels needing KNN fallback:  54,270



KNN-imputed dor_area_sqft for 907 parcels
KNN-imputed lycd_land_value for 54,270 parcels (no GMA zone or DOR area)


Capped lycd_land_value at market_value (improved parcels only): 22,173

GMA level used:
gma_level
L3     524536
knn     54270
L2        906
L1        102

All parcels have lycd_land_value: True


## Step 5: Summarize LYCD land base

In [6]:
total_lycd_land = gdf['lycd_land_value'].sum()
total_opa_land  = gdf['taxable_land'].sum()
print(f'Total LYCD land base:    ${total_lycd_land/1e9:.2f}B')
print(f'Total OPA taxable land:  ${total_opa_land/1e9:.2f}B')
print(f'Ratio LYCD/OPA:          {total_lycd_land/total_opa_land:.2f}x')
print()
print('LYCD land value by GMA level (median $):')
print(gdf.groupby('gma_level')['lycd_land_value'].median().sort_values(ascending=False).to_string())
print()
print('LYCD land value percentiles (all parcels):')
for p in [10, 25, 50, 75, 90, 99]:
    v = gdf['lycd_land_value'].quantile(p/100)
    print(f'  p{p:2d}: ${v:,.0f}')


Total LYCD land base:    $69.30B
Total OPA taxable land:  $36.62B
Ratio LYCD/OPA:          1.89x

LYCD land value by GMA level (median $):
gma_level
L2     826650.000000
L1     389700.000000
knn    110086.464000
L3      37671.535907

LYCD land value percentiles (all parcels):
  p10: $15,193
  p25: $23,742
  p50: $40,038
  p75: $69,969
  p90: $145,698
  p99: $939,600


## Step 6: Categorize parcels (same overrides as OPA model)

In [7]:
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement -> Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: abated improved parcels
GENUINE_VACANT_CODES = {'6', '12', '13'}
abated_mask = (
    (gdf['taxable_building'] <= 0) &
    (gdf['taxable_land'] > 0) &
    (~gdf['category_code'].isin(GENUINE_VACANT_CODES))
)
gdf.loc[abated_mask, 'PROPERTY_CATEGORY'] = 'Abated / Construction Exemption'

# Override 3: OPA-vacant with nonzero building value
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) &
    (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: fully exempt parcels
EXEMPT_CATEGORY_MAP = {k: v + ' â€” Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code']
    .map(EXEMPT_CATEGORY_MAP)
    .fillna('Other â€” Exempt')
)

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable: {(gdf["full_exmp"] == 0).sum():,}')
print()
print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

Total parcels: 579,814
Fully exempt: 44,116  |  Abated: 30,111  |  Improved vacant: 1,489  |  Taxable: 535,698

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential                    408137
Small Multi-Family (2-4 units)                37944
Abated / Construction Exemption               30111
Vacant Land                                   27925
Single Family Residential â€” Exempt          27059
Mixed Use                                     13399
Vacant Land â€” Exempt                        11860
Commercial                                     8523
Industrial                                     3485
Commercial â€” Exempt                          3280
Large Multi-Family (5+ units)                  2547
Improved Vacant Land                           1489
Small Multi-Family (2-4 units) â€” Exempt      1114
Other Residential                              1068
Office / Commercial Condo                       805
Large Multi-Family (5+ units) â€” Exempt        282
Indust

## Step 7: Post-Abatement Baseline Tax

Pre-shift baseline treats all active construction abatements as **expired**: the building value
in `exempt_building` is restored to the taxable base for abated parcels before computing `current_tax`.
This produces a **higher revenue baseline than FY2024 actuals** — the LVT reform is revenue-neutral
relative to this higher counterfactual baseline.


In [8]:
gdf['millage_rate'] = MILLAGE

# Post-abatement baseline: restore abated building values.
gdf['model_building'] = pd.to_numeric(gdf['taxable_building'], errors='coerce').fillna(0).clip(lower=0)

abated = gdf['PROPERTY_CATEGORY'] == 'Abated / Construction Exemption'
exempt_bldg  = pd.to_numeric(gdf['exempt_building'], errors='coerce').fillna(0)
market_val   = pd.to_numeric(gdf['market_value'],    errors='coerce').fillna(0)
tax_land     = pd.to_numeric(gdf['taxable_land'],     errors='coerce').fillna(0)
implied_bldg = (market_val - tax_land).clip(lower=0)
abated_bldg  = exempt_bldg.where(exempt_bldg > 0, implied_bldg)
gdf.loc[abated, 'model_building'] = abated_bldg[abated].values

n_exempt_bldg = int((abated & (exempt_bldg > 0)).sum())
n_fallback    = int((abated & (exempt_bldg <= 0)).sum())
abated_bldg_total = gdf.loc[abated, 'model_building'].sum()
print(f'Abated parcels using exempt_building:        {n_exempt_bldg:,}')
print(f'Abated parcels using market_value fallback:  {n_fallback:,}')
print(f'Total abated building base restored:         ${abated_bldg_total/1e9:.2f}B')
print()

# post_abatement_total: OPA taxable_land + model_building (abatements treated as expired)
gdf['post_abatement_total'] = (
    pd.to_numeric(gdf['taxable_land'], errors='coerce').fillna(0).clip(lower=0)
    + gdf['model_building']
).clip(lower=0)

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='post_abatement_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

actual_fy2024 = gdf['taxable_total'].mul(MILLAGE / 1000).sum()
print(f'Post-abatement baseline (city + school): ${current_revenue:,.0f}')
print(f'Actual FY2024 combined levy (modeled):   ${actual_fy2024:,.0f}')
print(f'Added by restoring abated buildings:     ${current_revenue - actual_fy2024:,.0f}')


Abated parcels using exempt_building:        27,984
Abated parcels using market_value fallback:  2,127
Total abated building base restored:         $15.00B



Post-abatement baseline (city + school): $2,061,178,704
Actual FY2024 combined levy (modeled):   $1,851,152,567
Added by restoring abated buildings:     $210,026,137


## Step 8: Build LYCD Reform Base (Post-Abatement)

Land value: GMA hierarchical LYCD (`lycd_land_value`).
Building value: `model_building` from Step 7 (restores `exempt_building` for abated parcels).
Revenue target: post-abatement baseline `current_tax` from Step 7.


In [9]:
gdf['model_land'] = gdf['lycd_land_value'].clip(lower=0)
# model_building already set in Step 7 (includes restored abated buildings).

print(f'Reform land base (LYCD):        ${gdf["model_land"].sum()/1e9:.2f}B')
print(f'Reform improvement base:         ${gdf["model_building"].sum()/1e9:.2f}B')
print(f'  of which abated bldg:          ${gdf.loc[abated, "model_building"].sum()/1e9:.2f}B')
print(f'OPA taxable land base:           ${pd.to_numeric(gdf["taxable_land"], errors="coerce").sum()/1e9:.2f}B')
print(f'OPA taxable building base:       ${pd.to_numeric(gdf["taxable_building"], errors="coerce").sum()/1e9:.2f}B')


Reform land base (LYCD):        $69.30B
Reform improvement base:         $110.63B
  of which abated bldg:          $15.00B
OPA taxable land base:           $36.62B
OPA taxable building base:       $95.62B


## Step 9: Revenue-neutral split-rate model (4:1 land:improvement)

In [10]:
taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='model_land',
    improvement_value_col='model_building',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine exempt parcels
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
exempt['taxable_land_value'] = 0.0
exempt['taxable_improvement_value'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} mills')
print(f'Improvement millage: {improvement_millage:.4f} mills')
print(f'Revenue check:       ${new_revenue:,.0f} (target: ${taxable["current_tax"].sum():,.0f})')
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Philadelphia â€” 4:1 Split-Rate Tax Impact (LYCD Land Values)')

Land millage:        26.3238 mills
Improvement millage: 6.5810 mills
Revenue check:       $2,061,178,704 (target: $2,061,178,704)




Philadelphia â€” 4:1 Split-Rate Tax Impact (LYCD Land Values)
                                 Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
                Single Family Residential 408137    $-24,563,467       -2.4%       $-60        $-235    2.2%     -14.2%            24.1%            56.2%
           Small Multi-Family (2-4 units)  37944    $-58,652,605      -26.8%    $-1,546        $-829  -18.1%     -25.3%            10.3%            76.5%
          Abated / Construction Exemption  30111    $-91,965,035      -36.2%    $-3,054        $-310  -16.8%     -21.5%            11.3%            72.9%
                              Vacant Land  27925    $253,652,381      744.9%     $9,083       $2,150 2016.1%     577.7%            97.8%             1.8%
     Single Family Residential â€” Exempt  27059              $0        0.0%         $0           $0    0.0%       0.0%             0.0%             0.0%
             

## Step 10: Census join

In [11]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f'Census join: {gdf["std_geoid"].notna().mean()*100:.1f}% matched')
        except concurrent.futures.TimeoutError:
            print('Census API timed out â€” skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


## Step 11: Export and visualize

In [12]:
out_df = save_standard_export(
    df=gdf,
    city=f'{CITY_NAME}_lycd_post_abatement',
    output_path=f'../../analysis/data/{CITY_NAME}_lycd_post_abatement.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    parcel_id_col='parcel_number',
)

from lvt.viz import create_city_report
create_city_report(out_df, f'{CITY_NAME}_lycd_post_abatement', show=False)
print('Done.')

  [warn] philadelphia_lycd_post_abatement: non-standard property categories (will be preserved): ['Abated / Construction Exemption', 'Commercial â€” Exempt', 'Hotel â€” Exempt', 'Improved Vacant Land', 'Industrial â€” Exempt', 'Large Multi-Family (5+ units) â€” Exempt', 'Mixed Use â€” Exempt', 'Office / Commercial Condo â€” Exempt', 'Other Commercial â€” Exempt', 'Other Residential â€” Exempt', 'Other â€” Exempt', 'Single Family Residential â€” Exempt', 'Small Multi-Family (2-4 units) â€” Exempt', 'Vacant Land â€” Exempt']


  ✓ philadelphia_lycd_post_abatement: 579,814 rows → ../../analysis/data/philadelphia_lycd_post_abatement.csv  [model: split_rate_4to1_lycd_post_abatement]


Done.
